In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"
OUTPUT_CSV = Path("data") / "sensitivity.csv"

df = pd.read_csv(INPUT_CSV)

print("Input shape:", df.shape)

# =============================================================================
# FUNCTIONS
# =============================================================================

def zscore(x):
    std = x.std(ddof=0)
    if std == 0:
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std


def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5


# =============================================================================
# 1. DISTRICT-MONTH AGGREGATION
# =============================================================================
# All variables first reduced from block → district per month

district_df = (
    df.groupby(["district", "timeperiod"], as_index=False)
      .agg(
          aged_pop=("sum_aged_population", "sum"),
          young_pop=("sum_young_population", "sum"),
          no_sanitation=("block_nosanitation_hhds_pct", "mean"),
          nco_5_9=("nco_5_9_percent_estimated", "mean"),
          pct_ncd=("pct_ncd", "mean")   # NEW VARIABLE ADDED
      )
)

# =============================================================================
# 2. MONTH-WISE Z-SCORES (ACROSS DISTRICTS)
# =============================================================================

sensitivity_vars = [
    "aged_pop",
    "young_pop",
    "no_sanitation",
    "nco_5_9",
    "pct_ncd"
]

for var in sensitivity_vars:
    district_df[var + "_z"] = (
        district_df.groupby("timeperiod")[var]
        .transform(zscore)
    )

# =============================================================================
# 3. BINNING (1–5 FOR EACH COMPONENT)
# =============================================================================

for var in sensitivity_vars:
    district_df[var + "_bin"] = district_df[var + "_z"].apply(classify)

# =============================================================================
# 4. COMPOSITE SENSITIVITY SCORE
# =============================================================================

bin_cols = [v + "_bin" for v in sensitivity_vars]

district_df["sensitivity_raw"] = district_df[bin_cols].sum(axis=1)

# =============================================================================
# 5. FINAL NORMALIZATION (MONTH-WISE)
# =============================================================================

district_df["sens_z"] = (
    district_df.groupby("timeperiod")["sensitivity_raw"]
    .transform(zscore)
)

district_df["sensitivity_class"] = district_df["sens_z"].apply(classify)

# =============================================================================
# 6. OUTPUT
# =============================================================================

output_cols = (
    ["district", "timeperiod"]
    + sensitivity_vars
    + [v + "_z" for v in sensitivity_vars]
    + bin_cols
    + ["sensitivity_raw", "sens_z", "sensitivity_class"]
)

district_df[output_cols].to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)

# =============================================================================
# 7. CHECKS
# =============================================================================

print("\nSensitivity distribution:")
print(district_df["sensitivity_class"].value_counts().sort_index())

print("\nPreview:")
print(district_df.head())

Input shape: (7222, 20)
Saved: data/sensitivity.csv

Sensitivity distribution:
sensitivity_class
1     46
2     92
3    414
4     92
5     46
Name: count, dtype: int64

Preview:
  district timeperiod   aged_pop  young_pop  no_sanitation  nco_5_9  pct_ncd  \
0   Anugul    2023_01  84749.045  90931.373      13.656656     79.0     20.0   
1   Anugul    2023_02  84749.045  90931.373      13.656656     79.0     20.0   
2   Anugul    2023_03  84749.045  90931.373      13.656656     79.0     20.0   
3   Anugul    2023_04  84749.045  90931.373      13.656656     79.0     20.0   
4   Anugul    2023_05  84749.045  90931.373      13.656656     79.0     20.0   

   aged_pop_z  young_pop_z  no_sanitation_z  nco_5_9_z  pct_ncd_z  \
0   -0.426815    -0.440308        -0.051004   0.217872  -0.524967   
1   -0.426815    -0.440308        -0.051004   0.217872  -0.524967   
2   -0.426815    -0.440308        -0.051004   0.217872  -0.524967   
3   -0.426815    -0.440308        -0.051004   0.217872  -0.524967